### Libraries

In [56]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

import torch
import torchinfo
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score


from pathlib import Path
Root = Path('.').absolute().parent
# DATA = Root/ r'C:\Users\Admin\Projects\ML Projects\ManipDetect\data'
DATA = Root/ r'C:\Users\krishnadas\Projects\ML Projects\ManipDetect\data'

In [5]:
def prepare_lstm_sequences(df, sequence:str, sequence_name:str, sequence_length:int):
    """Prepare sequences for LSTM training
    Choose the correct df based on the sequence type
    sequence: 'daily', 'hourly', or 'weekly'
    sequence_name: 'date', 'hour', or 'week'
    """
    assert sequence in ['daily', 'hourly', 'weekly'], \
        "Invalid sequence type. Choose from 'daily', 'hourly', or 'weekly'."
    assert sequence_name in ['date', 'hour', 'week'], \
        "Invalid sequence name. Choose from 'date', 'hour', or 'week'."
    assert sequence_length in [7, 24, 4], \
        "Invalid sequence length. Choose from 7 (daily), 24 (hourly), or 4 (weekly)."
    try:
        # df = label_manipulation_periods_sequence(df, sequence, sequence_name)
        df = df.sort_values(sequence_name).reset_index(drop=True)
        # Select feature columns (exclude date and target)
        feature_cols = [col for col in df.columns 
                        if col not in [sequence_name, 'is_manipulation','date', 'week']]
        # Create sequences
        X, y = [], []
        
        for i in range(sequence_length, len(df)):
            # Use previous 7 days to predict current day
            X.append(df[feature_cols].iloc[i-sequence_length:i].values)
            y.append(df['is_manipulation'].iloc[i])
        
    except Exception as e:
        print(f"Error preparing sequences: {e}")
        return None, None, None
    
    return np.array(X), np.array(y), feature_cols

In [6]:
# load the data
hourly_data = pd.read_pickle(DATA/'hourly_data.pkl')
daily_data = pd.read_pickle(DATA/'daily_data.pkl')
weekly_data = pd.read_pickle(DATA/'weekly_data.pkl')
wsb_data = pd.read_pickle(DATA/'df_wsb_data.pkl')

# X_hourly, y_hourly, feature_cols_hourly = prepare_lstm_sequences(hourly_data, sequence='hourly', sequence_name='hour', sequence_length=24)
# X_daily, y_daily, feature_cols_daily = prepare_lstm_sequences(daily_data, sequence='daily', sequence_name='date', sequence_length=7)
# X_weekly, y_weekly, feature_cols_weekly = prepare_lstm_sequences(weekly_data, sequence='weekly', sequence_name='week', sequence_length=4)

In [117]:
hourly_data.shape

(5922, 20)

In [89]:
# tranform the data to tensor format
hourly_tensor = torch.FloatTensor(X_hourly)
daily_tensor = torch.FloatTensor(X_daily)
weekly_tensor = torch.FloatTensor(X_weekly)

label_tensor = torch.FloatTensor(y_weekly)

In [120]:
print(hourly_tensor.shape)
print(daily_tensor.shape)
print(weekly_tensor.shape)
print(label_tensor.shape)


torch.Size([5898, 24, 17])
torch.Size([357, 7, 9])
torch.Size([49, 4, 4])
torch.Size([49])


In [21]:
def align_hierarchical_data(hourly_df, daily_df, weekly_df, show_details=False):
    """
    Align data at different temporal scales to create consistent batches
    
    Args:
        hourly_data: (5898, 24, 17) - Individual 24-hour periods
        daily_data: (357, 7, 9) - Individual 7-day periods
        weekly_data: (49, 4, 4) - Individual 4-week periods
        labels: (49,) - Weekly-level labels
    
    Returns:
        Aligned data where batch sizes are consistent
    """
    hourly_data, _, _ = prepare_lstm_sequences(hourly_df, sequence='hourly', sequence_name='hour', sequence_length=24)
    daily_data, _, _ = prepare_lstm_sequences(daily_df, sequence='daily', sequence_name='date', sequence_length=7)
    weekly_data, labels, _ = prepare_lstm_sequences(weekly_df, sequence='weekly', sequence_name='week', sequence_length=4)
    hourly_tensor = torch.FloatTensor(hourly_data)
    daily_tensor = torch.FloatTensor(daily_data)
    weekly_tensor = torch.FloatTensor(weekly_data)
    labels = torch.FloatTensor(labels)
    if show_details:
        print("Aligning hierarchical data...")
        print(f"Original shapes:")
        print(f"  Hourly: {hourly_tensor.shape}")
        print(f"  Daily: {daily_tensor.shape}")
        print(f"  Weekly: {weekly_tensor.shape}")
        print(f"  Labels: {labels.shape}")
    
    # The final batch size is determined by weekly data (smallest)
    num_weekly_samples = len(weekly_tensor)
    weeks_per_sample = 4
    days_per_week = 7
    hours_per_day = 24
    
    # Calculate required samples at each level
    required_daily_samples = num_weekly_samples * weeks_per_sample  # 49 * 4 = 196
    required_hourly_samples = required_daily_samples * days_per_week  # 196 * 7 = 1372
    if show_details:
        print(f"\nRequired samples for alignment:")
        print(f"  Hourly needed: {required_hourly_samples}")
        print(f"  Daily needed: {required_daily_samples}")
        print(f"  Weekly available: {num_weekly_samples}")
    
    # Check if we have enough data
    if len(hourly_tensor) < required_hourly_samples:
        print(f"⚠️  Warning: Not enough hourly data ({len(hourly_tensor)} < {required_hourly_samples})")
        # Use what we have and repeat if necessary
        hourly_aligned = hourly_tensor[:len(hourly_tensor)]
        # Pad with the last sample if needed
        if len(hourly_aligned) < required_hourly_samples:
            padding_needed = required_hourly_samples - len(hourly_aligned)
            last_sample = hourly_aligned[-1:].repeat(padding_needed, 1, 1)
            hourly_aligned = torch.cat([hourly_aligned, last_sample], dim=0)
    else:
        hourly_aligned = hourly_tensor[:required_hourly_samples]
    
    if len(daily_tensor) < required_daily_samples:
        print(f"⚠️  Warning: Not enough daily data ({len(daily_tensor)} < {required_daily_samples})")
        daily_aligned = daily_tensor[:len(daily_tensor)]
        if len(daily_aligned) < required_daily_samples:
            padding_needed = required_daily_samples - len(daily_aligned)
            last_sample = daily_aligned[-1:].repeat(padding_needed, 1, 1)
            daily_aligned = torch.cat([daily_aligned, last_sample], dim=0)
    else:
        daily_aligned = daily_tensor[:required_daily_samples]
    
    # Weekly data is already the right size
    weekly_aligned = weekly_tensor
    labels_aligned = labels
    
    # Reshape data for hierarchical processing
    # Reshape hourly data: (required_hourly_samples, 24, 17) -> (num_weekly_samples, 4*7*24, 17)
    # hourly_reshaped = hourly_aligned.view(num_weekly_samples, weeks_per_sample * days_per_week, hours_per_day, -1)
    hourly_reshaped = hourly_aligned.view(num_weekly_samples, weeks_per_sample * days_per_week * hours_per_day, -1)
    
    # Reshape daily data: (required_daily_samples, 7, 9) -> (num_weekly_samples, 4*7, 9)
    daily_reshaped = daily_aligned.view(num_weekly_samples, weeks_per_sample * days_per_week, -1)
    if show_details:
        print(f"\nAligned shapes:")
        print(f"  Hourly: {hourly_reshaped.shape}")  # (49, 672, 17) = 49 samples × 28 days × 24 hours
        print(f"  Daily: {daily_reshaped.shape}")    # (49, 28, 9) = 49 samples × 28 days
        print(f"  Weekly: {weekly_aligned.shape}")   # (49, 4, 4) = 49 samples × 4 weeks
        print(f"  Labels: {labels_aligned.shape}")   # (49,)
    
    return hourly_reshaped, daily_reshaped, weekly_aligned, labels_aligned


In [17]:
hourly_tensor, daily_tensor, weekly_tensor, labels=align_hierarchical_data(
        hourly_data, daily_data, weekly_data
    )

Aligning hierarchical data...
Original shapes:
  Hourly: torch.Size([5898, 24, 17])
  Daily: torch.Size([357, 7, 9])
  Weekly: torch.Size([49, 4, 4])
  Labels: torch.Size([49])

Required samples for alignment:
  Hourly needed: 1372
  Daily needed: 196
  Weekly available: 49

Aligned shapes:
  Hourly: torch.Size([49, 672, 17])
  Daily: torch.Size([49, 28, 9])
  Weekly: torch.Size([49, 4, 4])
  Labels: torch.Size([49])


### Hierarchical Dataset

In [ ]:
# This is not needed if we already have the data set in torch type
class HierarchicalLSTMDataset(Dataset):
    def __init__(self, hourly_data, daily_data, weekly_data, labels):
        self.hourly_data = torch.FloatTensor(hourly_data)
        self.daily_data = torch.FloatTensor(daily_data)
        self.weekly_data = torch.FloatTensor(weekly_data)
        self.labels = labels # labels correspond only to weekly_data

        self.length = min(len(hourly_data), len(daily_data), len(weekly_data))
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        return {
            'hourly': self.hourly_data[idx],
            'daily': self.daily_data[idx], 
            'weekly': self.weekly_data[idx],
            'label': self.labels[idx]
        }

### Hourly LSTM

In [47]:
class HourlyLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, self.hidden_size, self.num_layers, batch_first=True)

        # Attention mechanism to focus on most suspicious hours
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=4,
            dropout=dropout,
            batch_first=True
        )
        # Output projection to create hourly embedding
        self.output_projection = nn.Linear(hidden_size, hidden_size // 2)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size = x.size(0)
        # initialize the hidden state
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)

        #LSTM forward pass
        lstm_out, (hidden, cell) = self.lstm(x, (h0,c0))

        # Apply attention to focus on suspicious time periods
        attention_out, attention_weights = self.attention(lstm_out,lstm_out,lstm_out)

        # Global max pooling to capture strongest coordination signals
        pooled = torch.max(attention_out, dim=1)[0]  # (batch_size, hidden_size)
        
        # Project to embedding space
        hourly_embedding = self.output_projection(pooled)
        hourly_embedding = self.dropout(hourly_embedding)
        
        return hourly_embedding, attention_weights

In [48]:
# Create dummy hourly_data
batch_size = 10
hourly_features_size = 17
seq_len = 24

model = HourlyLSTM(input_size=hourly_features_size)
dummy_input = torch.randn(batch_size, seq_len, hourly_features_size)
hourly_embedding, attention_weights = model(dummy_input)
# Print the output shapes
print(f"Input shape: {dummy_input.shape}")
print(f"Hourly embedding shape: {hourly_embedding.shape}")  # Expected: (batch_size, hidden_size // 2)
print(f"Attention weights shape: {attention_weights.shape}")  # Expected: (batch_size, seq_len, seq_len)

torchinfo.summary(model, input_size=(batch_size, seq_len, hourly_features_size))

Input shape: torch.Size([10, 24, 17])
Hourly embedding shape: torch.Size([10, 32])
Attention weights shape: torch.Size([10, 24, 24])


Layer (type:depth-idx)                   Output Shape              Param #
HourlyLSTM                               [10, 32]                  --
├─LSTM: 1-1                              [10, 24, 64]              54,528
├─MultiheadAttention: 1-2                [10, 24, 64]              16,640
├─Linear: 1-3                            [10, 32]                  2,080
├─Dropout: 1-4                           [10, 32]                  --
Total params: 73,248
Trainable params: 73,248
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 13.11
Input size (MB): 0.02
Forward/backward pass size (MB): 0.13
Params size (MB): 0.23
Estimated Total Size (MB): 0.37

### Daily LSTM

In [49]:
class DailyLSTM(nn.Module):
    def __init__(self, daily_input_size, hourly_embedding_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        combined_input_size = daily_input_size + hourly_embedding_size
        # fusion layer to combine the daily features and hourly embeddings
        self.fusion_layer = nn.Sequential(nn.Linear(combined_input_size , combined_input_size),
                                        nn.ReLU(),
                                        nn.Dropout(dropout))
        self.lstm = nn.LSTM(combined_input_size , self.hidden_size, self.num_layers, batch_first=True)
        
        # Output projection
        self.output_projection = nn.Linear(hidden_size, hidden_size // 2)
        self.dropout = nn.Dropout(dropout)

    def forward(self, daily_features, hourly_embeddings):
        """
        daily_features: (batch_size, 7, daily_input_size)
        hourly_embeddings: (batch_size, 7, hourly_embedding_size)  
        Returns: (batch_size, hidden_size//2) - weekly embedding
        """
        batch_size = daily_features.size(0)
        
        # Combine daily features with hourly embeddings
        combined_input = torch.cat([daily_features, hourly_embeddings], dim=-1)
        
        # Apply fusion layer with residual connection
        fused_input = self.fusion_layer(combined_input)
        fused_input = fused_input + combined_input  # Residual connection
        
        # Initialize hidden states
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(daily_features.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(daily_features.device)
        
        # LSTM forward pass
        lstm_out, _ = self.lstm(fused_input, (h0, c0))
        
        # Take the last output (most recent day pattern)
        daily_embedding = self.output_projection(lstm_out[:, -1, :])
        daily_embedding = self.dropout(daily_embedding)
        
        return daily_embedding


In [50]:
batch_size = 1000
seq_len = 7
input_size = 17
hourly_embedding_size = 32
model = DailyLSTM(daily_input_size=input_size, hourly_embedding_size=hourly_embedding_size)

# Simulate input data
# Simulate input data for daily features and hourly embeddings
daily_features_shape = (batch_size, seq_len, input_size)
hourly_embeddings_shape = (batch_size, seq_len, hourly_embedding_size)

# Pass the inputs to the model
torchinfo.summary(model, input_size=(daily_features_shape, hourly_embeddings_shape))

Layer (type:depth-idx)                   Output Shape              Param #
DailyLSTM                                [1000, 32]                --
├─Sequential: 1-1                        [1000, 7, 49]             --
│    └─Linear: 2-1                       [1000, 7, 49]             2,450
│    └─ReLU: 2-2                         [1000, 7, 49]             --
│    └─Dropout: 2-3                      [1000, 7, 49]             --
├─LSTM: 1-2                              [1000, 7, 64]             62,720
├─Linear: 1-3                            [1000, 32]                2,080
├─Dropout: 1-4                           [1000, 32]                --
Total params: 67,250
Trainable params: 67,250
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 443.57
Input size (MB): 1.37
Forward/backward pass size (MB): 6.58
Params size (MB): 0.27
Estimated Total Size (MB): 8.23

### Weekly LSTM

In [51]:
class WeeklyLSTM(nn.Module):
    def __init__(self, weekly_input_size, daily_embedding_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # combined input size
        combined_input_size = weekly_input_size + daily_embedding_size

        # LSTM for processing weekly sequences
        self.lstm = nn.LSTM(input_size=combined_input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            dropout=dropout if num_layers>1 else 0,
                            batch_first=True)
        
        # Classification head
        self.classifier = nn.Sequential(nn.Linear(hidden_size, hidden_size//2),
                                        nn.ReLU(),
                                        nn.Dropout(dropout),
                                        nn.Linear(hidden_size//2, hidden_size//4),
                                        nn.ReLU(),
                                        nn.Dropout(dropout),
                                        nn.Linear(hidden_size//4, 1),
                                        nn.Sigmoid())
        
    def forward(self, weekly_features, daily_embeddings):
        """
        weekly_features: (batch_size, 4, weekly_input_size)
        daily_embeddings: (batch_size, 4, daily_embedding_size)
        Returns: (batch_size, 1) - manipulation probability
        """
        batch_size = weekly_features.size(0)

        # combine weekly features with daily embeddings
        combined_input = torch.cat([weekly_features, daily_embeddings], dim=-1)

        # initialize hidden state
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)

        # LSTM forward pass
        lstm_out,_ = self.lstm(combined_input, (h0,c0))

        # Take the last output for final prediction
        final_output = lstm_out[:, -1, :]

        # Classification
        manipulation_prob = self.classifier(final_output)

        return manipulation_prob


In [52]:
# Use correct input sizes based on your data
weekly_input_size = 4  # matches feature_cols_weekly
daily_embedding_size = 32  # should match output of DailyLSTM
seq_len = 4
batch_size = 10
model = WeeklyLSTM(weekly_input_size=weekly_input_size, daily_embedding_size=daily_embedding_size)
weekly_feature_shape = (batch_size, seq_len, weekly_input_size)
daily_embedding_shape = (batch_size, seq_len, daily_embedding_size)

torchinfo.summary(model, input_size=(weekly_feature_shape, daily_embedding_shape))

Layer (type:depth-idx)                   Output Shape              Param #
WeeklyLSTM                               [10, 1]                   --
├─LSTM: 1-1                              [10, 4, 64]               59,392
├─Sequential: 1-2                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 32]                  2,080
│    └─ReLU: 2-2                         [10, 32]                  --
│    └─Dropout: 2-3                      [10, 32]                  --
│    └─Linear: 2-4                       [10, 16]                  528
│    └─ReLU: 2-5                         [10, 16]                  --
│    └─Dropout: 2-6                      [10, 16]                  --
│    └─Linear: 2-7                       [10, 1]                   17
│    └─Sigmoid: 2-8                      [10, 1]                   --
Total params: 62,017
Trainable params: 62,017
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 2.40
Input size (MB): 0.01
Forward/backwar

In [99]:
# from torchviz import make_dot
# from IPython.display import Image, display

# model = WeeklyLSTM(weekly_input_size=10, daily_embedding_size=5)
# weekly_features = torch.randn(32, 4, 10)
# daily_embeddings = torch.randn(32, 4, 5)
# output = model(weekly_features, daily_embeddings)
# dot = make_dot(output, params=dict(model.named_parameters()))
# output_path = "WeeklyLSTM"
# dot.render(output_path, format="png")

# # Display the image in the notebook
# display(Image(filename=output_path + ".png"))

### Hierarchical LSTM

In [42]:
class HierarchicalManipulationDetector(nn.Module):
    """
    Complete hierarchical LSTM model for market manipulation detection
    """
    def __init__(self, hourly_features, daily_features, weekly_features, hidden_size=64, dropout=0.2):
        super(HierarchicalManipulationDetector, self).__init__()
        
        # Initialize the three hierarchical levels
        self.hourly_lstm = HourlyLSTM(
            input_size=hourly_features,
            hidden_size=hidden_size,
            dropout=dropout
        )
        
        self.daily_lstm = DailyLSTM(
            daily_input_size=daily_features,
            hourly_embedding_size=hidden_size // 2,
            hidden_size=hidden_size,
            dropout=dropout
        )
        
        self.weekly_lstm = WeeklyLSTM(
            weekly_input_size=weekly_features,
            daily_embedding_size=hidden_size // 2,
            hidden_size=hidden_size,
            dropout=dropout
        )

    def forward(self, hourly_data, daily_data, weekly_data):
        """
        Forward pass through all three hierarchical levels
        
        hourly_data: (batch_size, 24, hourly_features) - 24 hours
        daily_data: (batch_size, 28, daily_features) - 28 days (4 weeks × 7 days) 
        weekly_data: (batch_size, 4, weekly_features) - 4 weeks
        
        Returns:
        - manipulation_probability: (batch_size, 1)
        - attention_weights: List of attention weights for interpretability
        """
        batch_size = hourly_data.size(0)
        
         # Process hourly data for each day
        hourly_embeddings = []
        attention_weights_list = []
        
        # We have 28 days of hourly data (4 weeks × 7 days)
        for day_idx in range(28):
            # Extract 24 hours for this day
            start_hour = day_idx * 24
            end_hour = start_hour + 24
            day_hourly_data = hourly_data[:, start_hour:end_hour, :]
            
            # Process through hourly LSTM
            hourly_emb, attention_weights = self.hourly_lstm(day_hourly_data)
            hourly_embeddings.append(hourly_emb)
            attention_weights_list.append(attention_weights)
        
        # Stack hourly embeddings: (batch_size, 28, hourly_embedding_size)
        hourly_embeddings = torch.stack(hourly_embeddings, dim=1)
        
        # Process daily data with hourly embeddings
        daily_embeddings = []
        
        # We have 4 weeks, process each week (7 days) separately
        for week_idx in range(4):
            # Extract 7 days for this week
            start_day = week_idx * 7
            end_day = start_day + 7
            
            week_daily_data = daily_data[:, start_day:end_day, :]
            week_hourly_emb = hourly_embeddings[:, start_day:end_day, :]
            
            daily_emb = self.daily_lstm(week_daily_data, week_hourly_emb)
            daily_embeddings.append(daily_emb)
        
        # Stack daily embeddings: (batch_size, 4, daily_embedding_size)
        daily_embeddings = torch.stack(daily_embeddings, dim=1)
        
        # Final prediction through weekly LSTM
        manipulation_probability = self.weekly_lstm(weekly_data, daily_embeddings)
        
        return manipulation_probability, attention_weights_list

In [124]:
import torch

batch_size = 49
hourly_features = 17
daily_features = 9
weekly_features = 4

model = HierarchicalManipulationDetector(
    hourly_features=hourly_features,
    daily_features=daily_features,
    weekly_features=weekly_features
)

# Correct input dimensions:
# hourly_data: 7 days × 24 hours = 168 time steps
example_hourly = torch.randn(batch_size, 672, hourly_features)  # 24 hours
# daily_data: 7 days worth of daily features
example_daily = torch.randn(batch_size, 28, daily_features)      # 7 days
# weekly_data: 4 weeks worth of weekly features  
example_weekly = torch.randn(batch_size, 4, weekly_features)    # 4 weeks

print(f"Input shapes:")
print(f"Hourly data: {example_hourly.shape}")
print(f"Daily data: {example_daily.shape}")
print(f"Weekly data: {example_weekly.shape}")

torchinfo.summary(model, input_data=(example_hourly, example_daily, example_weekly))

Input shapes:
Hourly data: torch.Size([49, 672, 17])
Daily data: torch.Size([49, 28, 9])
Weekly data: torch.Size([49, 4, 4])


Layer (type:depth-idx)                   Output Shape              Param #
HierarchicalManipulationDetector         [49, 1]                   --
├─HourlyLSTM: 1-1                        [49, 32]                  --
│    └─LSTM: 2-1                         [49, 24, 64]              54,528
│    └─MultiheadAttention: 2-2           [49, 24, 64]              16,640
│    └─Linear: 2-3                       [49, 32]                  2,080
│    └─Dropout: 2-4                      [49, 32]                  --
├─HourlyLSTM: 1-2                        [49, 32]                  (recursive)
│    └─LSTM: 2-5                         [49, 24, 64]              (recursive)
│    └─MultiheadAttention: 2-6           [49, 24, 64]              (recursive)
│    └─Linear: 2-7                       [49, 32]                  (recursive)
│    └─Dropout: 2-8                      [49, 32]                  --
├─HourlyLSTM: 1-3                        [49, 32]                  (recursive)
│    └─LSTM: 2-9             

## Why Loop Over 4 Weeks in HierarchicalManipulationDetector?

The loop over 4 weeks in your `HierarchicalManipulationDetector` is designed to create **4 weekly embeddings** that represent different time periods for the `WeeklyLSTM` to process.

### Architectural Flow:
1. **HourlyLSTM**: Processes 24 hours of data → produces 1 hourly embedding
2. **DailyLSTM**: Processes 7 days of data + hourly embeddings → produces 1 daily embedding  
3. **WeeklyLSTM**: Needs 4 weekly embeddings to process 4 weeks of temporal patterns → produces final prediction

The 4-week loop creates these 4 daily embeddings that represent weekly patterns for temporal analysis.

### Current Implementation Issue ⚠️

However, there's a **critical flaw** in your current approach: you're using the **same `daily_data` and `hourly_emb` for all 4 weeks**. This means all 4 weekly embeddings will be identical!

### What Should Happen Instead:
1. You should have **4 weeks × 7 days = 28 days** of daily data
2. Each week should get its **own 7-day slice** of daily data
3. Each week should get its **own hourly embedding** (or slice of hourly embeddings)

In [128]:
class AlignedHierarchicalDataset(Dataset):
    """
    Dataset with aligned hierarchical data
    """
    def __init__(self, hourly_data, daily_data, weekly_data, labels):
        """
        All inputs should have the same batch size (first dimension)
        """
        self.hourly_data = torch.FloatTensor(hourly_data)
        self.daily_data = torch.FloatTensor(daily_data)
        self.weekly_data = torch.FloatTensor(weekly_data)
        self.labels = torch.FloatTensor(labels)
        
        # Validate all have same batch size
        batch_size = len(hourly_data)
        assert len(daily_data) == batch_size, f"Daily data batch size {len(daily_data)} != {batch_size}"
        assert len(weekly_data) == batch_size, f"Weekly data batch size {len(weekly_data)} != {batch_size}"
        assert len(labels) == batch_size, f"Labels batch size {len(labels)} != {batch_size}"
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'hourly': self.hourly_data[idx],
            'daily': self.daily_data[idx],
            'weekly': self.weekly_data[idx],
            'label': self.labels[idx]
        }

In [132]:
# Example usage function
def prepare_your_data_for_training(hourly_tensor, daily_tensor, weekly_tensor, labels_tensor):
    """
    Complete data preparation pipeline
    """
    print("Preparing data for hierarchical LSTM training...")
    
    # Step 1: Align the data
    hourly_aligned, daily_aligned, weekly_aligned, labels_aligned = align_hierarchical_data(
        hourly_tensor, daily_tensor, weekly_tensor, labels_tensor
    )
    
    # Step 2: Create dataset
    dataset = AlignedHierarchicalDataset(
        hourly_aligned, daily_aligned, weekly_aligned, labels_aligned
    )
    
    # Step 3: Create data loaders
    from torch.utils.data import random_split
    
    train_size = int(0.7 * len(dataset))
    val_size = int(0.15 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)  # Small batch size
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    
    print(f"Created data loaders:")
    print(f"  Train: {len(train_dataset)} samples")
    print(f"  Val: {len(val_dataset)} samples") 
    print(f"  Test: {len(test_dataset)} samples")
    
    return train_loader, val_loader, test_loader


In [133]:

# Main execution example
if __name__ == "__main__":
    # Your original data shapes
    hourly_data = hourly_tensor  # Your actual data
    daily_data = daily_tensor     # Your actual data  
    weekly_data = weekly_tensor     # Your actual data
    labels = label_tensor      # Your actual labels
    
    # Prepare data
    train_loader, val_loader, test_loader = prepare_your_data_for_training(
        hourly_data, daily_data, weekly_data, labels
    )
    
    # Initialize model
    model = HierarchicalManipulationDetector(
        hourly_features=17,
        daily_features=9,
        weekly_features=4,
        hidden_size=64
    )
    
    # Test forward pass
    for batch in train_loader:
        hourly = batch['hourly']
        daily = batch['daily'] 
        weekly = batch['weekly']
        labels = batch['label']
        
        print(f"Batch shapes:")
        print(f"  Hourly: {hourly.shape}")
        print(f"  Daily: {daily.shape}")
        print(f"  Weekly: {weekly.shape}")
        print(f"  Labels: {labels.shape}")
        
        # Forward pass
        predictions, attention_weights = model(hourly, daily, weekly)
        print(f"  Predictions: {predictions.shape}")
        
        break  # Just test one batch
    
    print("✓ Data alignment and forward pass successful!")

Preparing data for hierarchical LSTM training...
Aligning hierarchical data...
Original shapes:
  Hourly: torch.Size([5898, 24, 17])
  Daily: torch.Size([357, 7, 9])
  Weekly: torch.Size([49, 4, 4])
  Labels: torch.Size([49])

Required samples for alignment:
  Hourly needed: 1372
  Daily needed: 196
  Weekly available: 49

Aligned shapes:
  Hourly: torch.Size([49, 672, 17])
  Daily: torch.Size([49, 28, 9])
  Weekly: torch.Size([49, 4, 4])
  Labels: torch.Size([49])
Created data loaders:
  Train: 34 samples
  Val: 7 samples
  Test: 8 samples
Batch shapes:
  Hourly: torch.Size([8, 672, 17])
  Daily: torch.Size([8, 28, 9])
  Weekly: torch.Size([8, 4, 4])
  Labels: torch.Size([8])
  Predictions: torch.Size([8, 1])
✓ Data alignment and forward pass successful!


In [36]:
# Create dummy hourly_data
batch_size = 10
weekly_features = 9
daily_data = torch.randn(batch_size, 7, weekly_features)
print(daily_data)

tensor([[[-1.4642e+00, -8.0803e-01,  1.2871e-01,  4.1248e-01,  2.1352e-01,
           2.6484e-01,  1.8079e+00, -1.9777e-01, -7.6337e-01],
         [ 7.6592e-01,  1.4632e-01, -6.5169e-01,  3.4402e-01, -2.4563e-01,
           7.2448e-02,  1.0885e-01, -7.7543e-02,  7.1240e-01],
         [-7.3368e-01,  7.5798e-01,  6.0691e-01,  8.1476e-01, -1.6105e-01,
           1.5402e-01, -8.7258e-01, -2.6947e-01,  1.0030e+00],
         [-7.6139e-01, -6.5371e-02, -2.0847e-02,  4.2677e-02, -5.9214e-01,
           2.3677e-01,  2.2309e+00, -4.6559e-01,  1.9490e+00],
         [ 2.3717e+00,  4.6234e-01, -1.1360e+00, -6.7226e-01,  4.5363e-01,
          -1.5648e+00, -2.4321e-01, -1.7069e+00, -4.1168e-01],
         [ 5.5099e-01,  1.1322e+00, -2.2869e-01,  3.1872e-01, -1.8183e+00,
           3.5638e-01,  7.4548e-02,  2.4782e+00, -8.3228e-01],
         [-1.3247e+00,  8.9757e-01, -1.7911e+00, -1.4961e+00, -3.0739e-01,
           1.2043e+00, -4.1228e-01,  3.5379e-01,  1.2593e+00]],

        [[ 8.5455e-01,  2.6278e-

## Training

In [53]:
def prepare_hierarchical_dataset(hourly_data, daily_data, weekly_data, 
                                test_split=0.2, validation_split=0.2, random_seed=42):
    np.random.seed(random_seed)
    hourly_tensor, daily_tensor, weekly_tensor, labels = align_hierarchical_data(
        hourly_data, daily_data, weekly_data
    )

    # Calculate split sizes
    total_samples = len(labels)
    test_size = int(test_split * total_samples)
    remaining_samples = total_samples - test_size
    val_size = int(validation_split * remaining_samples)
    train_size = remaining_samples - val_size
    print(f"\nSplit Configuration:")
    print(f"   Total samples: {total_samples}")
    print(f"   Train samples: {train_size} ({train_size/total_samples:.1%})")
    print(f"   Validation samples: {val_size} ({val_size/total_samples:.1%})")
    print(f"   Test samples: {test_size} ({test_size/total_samples:.1%})")
    
    indices = torch.randperm(total_samples)
        
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]
    
    # Train split
    train_hourly = hourly_tensor[train_indices]
    train_daily = daily_tensor[train_indices]
    train_weekly = weekly_tensor[train_indices]
    train_labels = labels[train_indices]
    
    # Validation split
    val_hourly = hourly_tensor[val_indices]
    val_daily = daily_tensor[val_indices]
    val_weekly = weekly_tensor[val_indices]
    val_labels = labels[val_indices]
    
    # Test split
    test_hourly = hourly_tensor[test_indices]
    test_daily = daily_tensor[test_indices]
    test_weekly = weekly_tensor[test_indices]
    test_labels = labels[test_indices]

    # Step 4: Analyze label distribution
    train_pos = train_labels.sum().item()
    val_pos = val_labels.sum().item()
    test_pos = test_labels.sum().item()
    
    print(f"\nLabel Distribution:")
    print(f"   Train: {train_pos}/{len(train_labels)} positive ({train_pos/len(train_labels):.1%})")
    print(f"   Val:   {val_pos}/{len(val_labels)} positive ({val_pos/len(val_labels):.1%})")
    print(f"   Test:  {test_pos}/{len(test_labels)} positive ({test_pos/len(test_labels):.1%})")

    # Step 5: Create metadata
    split_info = {
        'total_samples': total_samples,
        'train_size': train_size,
        'val_size': val_size,
        'test_size': test_size,
        'train_positive_rate': train_pos / len(train_labels),
        'val_positive_rate': val_pos / len(val_labels),
        'test_positive_rate': test_pos / len(test_labels),
        'random_seed': random_seed,
        'original_shapes': {
            'hourly': hourly_data.shape,
            'daily': daily_data.shape,
            'weekly': weekly_data.shape
        },
        'aligned_shapes': {
            'hourly': hourly_tensor.shape,
            'daily': daily_tensor.shape,
            'weekly': weekly_tensor.shape
        }
    }
    
    # Step 6: Package results
    dataset_splits = {
        'train': (train_hourly, train_daily, train_weekly, train_labels),
        'val': (val_hourly, val_daily, val_weekly, val_labels),
        'test': (test_hourly, test_daily, test_weekly, test_labels),
        'info': split_info
    }
    
    print("Dataset preparation completed!")
    return dataset_splits
    

In [33]:
dataset_splits=prepare_hierarchical_dataset(hourly_data, daily_data, weekly_data)


Split Configuration:
   Total samples: 49
   Train samples: 32 (65.3%)
   Validation samples: 8 (16.3%)
   Test samples: 9 (18.4%)

Label Distribution:
   Train: 1.0/32 positive (3.1%)
   Val:   1.0/8 positive (12.5%)
   Test:  0.0/9 positive (0.0%)
Dataset preparation completed!


In [57]:
def train_hierarchical_model(train_data, val_data, model_config=None, training_config=None):
    """
    Train the hierarchical LSTM model on prepared datasets
    
    Args:
        train_data: Tuple of (hourly, daily, weekly, labels) for training
        val_data: Tuple of (hourly, daily, weekly, labels) for validation
        model_config: Dictionary with model hyperparameters
        training_config: Dictionary with training hyperparameters
    
    Returns:
        trained_model, training_history
    """
    
    print("🚀 Training Hierarchical LSTM Model")
    print("=" * 40)
    
    # Unpack data
    train_hourly, train_daily, train_weekly, train_labels = train_data
    val_hourly, val_daily, val_weekly, val_labels = val_data
    
    # Default configurations
    if model_config is None:
        model_config = {
            'hourly_features': 17,
            'daily_features': 9,
            'weekly_features': 4,
            'hidden_size': 32,
            'dropout': 0.2
        }
    
    if training_config is None:
        training_config = {
            'epochs': 50,
            'learning_rate': 0.001,
            'patience': 10,
            'lr_patience': 5,
            'lr_factor': 0.5,
            'verbose': True
        }
    
    print(f"📋 Model Configuration: {model_config}")
    print(f"📋 Training Configuration: {training_config}")
    
    # Setup device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🖥️  Using device: {device}")
    
    # Initialize model
    model = HierarchicalManipulationDetector(**model_config)
    model.to(device)
    
    print(f"🧠 Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
    
    # Move data to device
    train_hourly = train_hourly.to(device)
    train_daily = train_daily.to(device)
    train_weekly = train_weekly.to(device)
    train_labels = train_labels.to(device)
    
    val_hourly = val_hourly.to(device)
    val_daily = val_daily.to(device)
    val_weekly = val_weekly.to(device)
    val_labels = val_labels.to(device)
    
    # Setup training components
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=training_config['learning_rate'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', 
        patience=training_config['lr_patience'], 
        factor=training_config['lr_factor'] 
        # verbose=training_config['verbose']
    )
    
    # Training tracking
    training_history = {
        'train_loss': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': [],
        'learning_rates': []
    }
    
    # Early stopping setup
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    print(f"\n🏋️ Starting training for {training_config['epochs']} epochs...")
    print("-" * 60)
    
    # Training loop
    for epoch in range(training_config['epochs']):
        # Training phase
        model.train()
        train_predictions, _ = model(train_hourly, train_daily, train_weekly)
        train_predictions = train_predictions.squeeze()
        
        train_loss = criterion(train_predictions, train_labels)
        
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()
        
        train_predicted_classes = (train_predictions > 0.5).float()
        train_accuracy = (train_predicted_classes == train_labels).float().mean()
        
        # Validation phase
        model.eval()
        with torch.no_grad():
            val_predictions, _ = model(val_hourly, val_daily, val_weekly)
            val_predictions = val_predictions.squeeze()
            val_loss = criterion(val_predictions, val_labels)
            
            val_predicted_classes = (val_predictions > 0.5).float()
            val_accuracy = (val_predicted_classes == val_labels).float().mean()
            
            # Detailed validation metrics
            val_true_np = val_labels.cpu().numpy()
            val_pred_np = val_predicted_classes.cpu().numpy()
            
            val_precision = precision_score(val_true_np, val_pred_np, zero_division=0)
            val_recall = recall_score(val_true_np, val_pred_np, zero_division=0)
            val_f1 = f1_score(val_true_np, val_pred_np, zero_division=0)
        
        # Update scheduler
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Store metrics
        training_history['train_loss'].append(train_loss.item())
        training_history['train_accuracy'].append(train_accuracy.item())
        training_history['val_loss'].append(val_loss.item())
        training_history['val_accuracy'].append(val_accuracy.item())
        training_history['val_precision'].append(val_precision)
        training_history['val_recall'].append(val_recall)
        training_history['val_f1'].append(val_f1)
        training_history['learning_rates'].append(current_lr)
        
        # Print progress
        if training_config['verbose']:
            print(f"Epoch {epoch+1:3d}/{training_config['epochs']}: "
                  f"Train Loss={train_loss.item():.4f}, Train Acc={train_accuracy.item():.4f} | "
                  f"Val Loss={val_loss.item():.4f}, Val Acc={val_accuracy.item():.4f}, "
                  f"Val F1={val_f1:.4f}, LR={current_lr:.2e}")
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            if training_config['verbose']:
                print(f"      → New best validation loss: {val_loss.item():.4f}")
        else:
            patience_counter += 1
            if patience_counter >= training_config['patience']:
                print(f"\n⏱️  Early stopping triggered after {epoch+1} epochs")
                break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"✅ Loaded best model with validation loss: {best_val_loss:.4f}")
    
    print("🎉 Training completed successfully!")
    return model, training_history

In [44]:
train_data = dataset_splits['train']
val_data = dataset_splits['val']
test_data = dataset_splits['test']

In [58]:
model, training_history = train_hierarchical_model(train_data, val_data)

🚀 Training Hierarchical LSTM Model
📋 Model Configuration: {'hourly_features': 17, 'daily_features': 9, 'weekly_features': 4, 'hidden_size': 32, 'dropout': 0.2}
📋 Training Configuration: {'epochs': 50, 'learning_rate': 0.001, 'patience': 10, 'lr_patience': 5, 'lr_factor': 0.5, 'verbose': True}
🖥️  Using device: cpu
🧠 Model initialized with 52,939 parameters

🏋️ Starting training for 50 epochs...
------------------------------------------------------------
Epoch   1/50: Train Loss=0.7165, Train Acc=0.0312 | Val Loss=0.7117, Val Acc=0.1250, Val F1=0.2222, LR=1.00e-03
      → New best validation loss: 0.7117
Epoch   1/50: Train Loss=0.7165, Train Acc=0.0312 | Val Loss=0.7117, Val Acc=0.1250, Val F1=0.2222, LR=1.00e-03
      → New best validation loss: 0.7117
Epoch   2/50: Train Loss=0.7155, Train Acc=0.0312 | Val Loss=0.7107, Val Acc=0.1250, Val F1=0.2222, LR=1.00e-03
      → New best validation loss: 0.7107
Epoch   2/50: Train Loss=0.7155, Train Acc=0.0312 | Val Loss=0.7107, Val Acc=0.125

In [59]:
def evaluate_model(model, test_data, save_path=None, verbose=True):
    """
    Evaluate the trained model on test data
    
    Args:
        model: Trained model
        test_data: Tuple of (hourly, daily, weekly, labels) for testing
        save_path: Optional path to save the model
        verbose: Whether to print detailed results
    
    Returns:
        Dictionary with evaluation metrics
    """
    
    if verbose:
        print("📊 Evaluating Model on Test Set")
        print("=" * 30)
    
    test_hourly, test_daily, test_weekly, test_labels = test_data
    
    # Ensure data is on the same device as model
    device = next(model.parameters()).device
    test_hourly = test_hourly.to(device)
    test_daily = test_daily.to(device)
    test_weekly = test_weekly.to(device)
    test_labels = test_labels.to(device)
    
    # Evaluation
    model.eval()
    with torch.no_grad():
        test_predictions, test_attention = model(test_hourly, test_daily, test_weekly)
        test_predictions = test_predictions.squeeze()
        
        test_predicted_classes = (test_predictions > 0.5).float()
        test_accuracy = (test_predicted_classes == test_labels).float().mean()
        
        # Convert to numpy for sklearn metrics
        test_true_np = test_labels.cpu().numpy()
        test_pred_np = test_predicted_classes.cpu().numpy()
        test_pred_probs = test_predictions.cpu().numpy()
    
    # Calculate comprehensive metrics
    test_precision = precision_score(test_true_np, test_pred_np, zero_division=0)
    test_recall = recall_score(test_true_np, test_pred_np, zero_division=0)
    test_f1 = f1_score(test_true_np, test_pred_np, zero_division=0)
    
    evaluation_results = {
        'accuracy': test_accuracy.item(),
        'precision': test_precision,
        'recall': test_recall,
        'f1_score': test_f1,
        'predictions': test_pred_probs,
        'true_labels': test_true_np,
        'predicted_classes': test_pred_np,
        'attention_weights': test_attention
    }
    
    if verbose:
        print(f"🎯 Test Accuracy: {test_accuracy.item():.4f}")
        print(f"📈 Test Precision: {test_precision:.4f}")
        print(f"📈 Test Recall: {test_recall:.4f}")
        print(f"📈 Test F1-Score: {test_f1:.4f}")
        
        print("\n📋 Detailed Classification Report:")
        print(classification_report(test_true_np, test_pred_np, target_names=['Normal', 'Manipulation']))
        
        print("\n🔍 Sample Predictions:")
        for i in range(min(5, len(test_labels))):
            true_label = "Manipulation" if test_true_np[i] > 0.5 else "Normal"
            pred_label = "Manipulation" if test_pred_np[i] > 0.5 else "Normal"
            confidence = test_pred_probs[i]
            print(f"   Sample {i+1}: True={true_label:12}, Predicted={pred_label:12} (confidence={confidence:.3f})")
    
    # Save model if requested
    if save_path:
        torch.save(model.state_dict(), save_path)
        if verbose:
            print(f"\n💾 Model saved to: {save_path}")
    
    return evaluation_results

def plot_training_history(history):
    """Plot comprehensive training curves"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss curves
    axes[0, 0].plot(history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[0, 0].plot(history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy curves
    axes[0, 1].plot(history['train_accuracy'], 'b-', label='Train Accuracy', linewidth=2)
    axes[0, 1].plot(history['val_accuracy'], 'r-', label='Validation Accuracy', linewidth=2)
    axes[0, 1].set_title('Training and Validation Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Validation metrics
    axes[1, 0].plot(history['val_precision'], 'g-', label='Precision', linewidth=2)
    axes[1, 0].plot(history['val_recall'], 'orange', label='Recall', linewidth=2)
    axes[1, 0].plot(history['val_f1'], 'purple', label='F1-Score', linewidth=2)
    axes[1, 0].set_title('Validation Metrics')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Score')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning rate schedule
    axes[1, 1].semilogy(history['learning_rates'], 'purple', linewidth=2)
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(y_true, y_pred):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Manipulation'],
                yticklabels=['Normal', 'Manipulation'])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

In [61]:
# Step 2: Train model
trained_model, training_history = train_hierarchical_model(
    dataset_splits['train'], dataset_splits['val'])


🚀 Training Hierarchical LSTM Model
📋 Model Configuration: {'hourly_features': 17, 'daily_features': 9, 'weekly_features': 4, 'hidden_size': 32, 'dropout': 0.2}
📋 Training Configuration: {'epochs': 50, 'learning_rate': 0.001, 'patience': 10, 'lr_patience': 5, 'lr_factor': 0.5, 'verbose': True}
🖥️  Using device: cpu
🧠 Model initialized with 52,939 parameters

🏋️ Starting training for 50 epochs...
------------------------------------------------------------
Epoch   1/50: Train Loss=0.7323, Train Acc=0.0312 | Val Loss=0.7251, Val Acc=0.1250, Val F1=0.2222, LR=1.00e-03
      → New best validation loss: 0.7251
Epoch   1/50: Train Loss=0.7323, Train Acc=0.0312 | Val Loss=0.7251, Val Acc=0.1250, Val F1=0.2222, LR=1.00e-03
      → New best validation loss: 0.7251
Epoch   2/50: Train Loss=0.7318, Train Acc=0.0312 | Val Loss=0.7221, Val Acc=0.1250, Val F1=0.2222, LR=1.00e-03
      → New best validation loss: 0.7221
Epoch   2/50: Train Loss=0.7318, Train Acc=0.0312 | Val Loss=0.7221, Val Acc=0.125

In [62]:
# Step 3: Evaluate model
evaluation_results = evaluate_model(
    model, dataset_splits['test'], verbose=True
)
# Step 4: Generate visualizations
plot_training_history(training_history)
plot_confusion_matrix(evaluation_results['true_labels'], evaluation_results['predicted_classes'])
    
# Step 5: Return comprehensive results
results = {
    'model': trained_model,
    'training_history': training_history,
    'evaluation_results': evaluation_results,
    'dataset_info': dataset_splits['info']
    }

📊 Evaluating Model on Test Set
🎯 Test Accuracy: 1.0000
📈 Test Precision: 0.0000
📈 Test Recall: 0.0000
📈 Test F1-Score: 0.0000

📋 Detailed Classification Report:
🎯 Test Accuracy: 1.0000
📈 Test Precision: 0.0000
📈 Test Recall: 0.0000
📈 Test F1-Score: 0.0000

📋 Detailed Classification Report:


ValueError: Number of classes, 1, does not match size of target_names, 2. Try specifying the labels parameter